# 08 — On-Device Model Benchmark

**Project:** IT22638168 — MaternaLink FER track
**Pipeline position:** step 8 of 8 · see `../README.md`

## What this notebook is

A **MODEL-level** benchmark of the four TFLite variants produced by notebook 07
(`float32`, `float16`, `dynint8`, `fullint8`). It does **not** read a single labelled
image and it never touches the TEST split — every accuracy number quoted here is carried
over from notebook 07 (VALIDATION).

### Metrics this notebook can produce

- **P-12** — TFLite file size on disk (all four variants). **CLOSED here.**
- **P-02** — FER inference latency around the TFLite `invoke`, **only** when a physical
  Android device and an `arm64-v8a` `benchmark_model` binary are present (Mode A).

### Metrics this notebook cannot produce

- **P-01** (face detection), **P-03** (preprocessing), **P-04** (end-to-end
  camera→mood), **P-14** (peak app memory) — these **require the Android application,
  which does not exist yet**. They are recorded as UNFILLED / "not measurable at this
  phase — requires the Android application".

## Two-mode design (auto-detected at runtime)

- **MODE A** — an authorised Android device is visible to `adb` **and** a usable
  `benchmark_model` binary for `arm64-v8a` is available. P-02 is measured on-device.
- **MODE B** — no device / no `adb` / no benchmark binary. **This is the expected
  outcome now and is a correct result, not a failure.** P-02 stays OPEN, Phase-3 exit
  stays blocked on it, and the notebook emits a device-benchmark protocol for whoever
  runs the device later. No x86 numbers are ever substituted for device latency.

## Before running

Read `../../../docs/system/MOOD_STATE_SPEC.md` (label space) and
`PERFORMANCE_BENCHMARK_PLAN.md` §4.3 (P-12 definition). Emulator / development-machine
timings are NEVER reported as device results. The Proposal's 3-second end-to-end target
is a *design target*, not a measured value, and is never reported as measured.

## Status

Not yet run.


In [1]:
"""Run metadata — every training/evaluation notebook records its own run.

This is the project's experiment tracking (Gate 1A resolution): no separate
infrastructure, the notebook is responsible. Satisfies NFR-12.
"""
import json, os, time, random

RUN = {
    "run_id":          time.strftime("run_%Y%m%d_%H%M%S"),
    "timestamp":       time.strftime("%Y-%m-%d %H:%M:%S"),
    "notebook":        "08_mobile_benchmark",
    "dataset_version": None,   # set once data/processed/ is written
    "model_version":   None,   # set when a model is saved
    "hyperparameters": {},    # lr, batch_size, epochs, augmentation...
    "random_seed":     42,
    "metrics":         {},    # accuracy, macro_f1, per_class...
    "notes":           "",
}

random.seed(RUN["random_seed"])

def save_run(run=RUN, outdir="../outputs"):
    """Write the run record. Call at the END of the notebook."""
    os.makedirs(outdir, exist_ok=True)
    path = os.path.join(outdir, run["run_id"] + ".json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(run, f, indent=2)
    print("saved:", path)
    return path

RUN["run_id"]


'run_20260828_181803'

---

## Work starts here


## 0. Environment setup — repo root, artifact dirs, byte tracker

Same portability logic as notebooks 03–07: `find_repo_root()` walks up for
`ml/fer/notebooks` and never depends on the CWD. This notebook reads no images, so there
is no `DATA_ROOT` resolution. All artifacts land directly in the repo under
`ml/fer/outputs/` and `ml/fer/plots/nb08/`.


In [2]:
import os
import sys
import time

# --- repo root: walk up until ml/fer/notebooks is found ----------------------
def find_repo_root(start=None, max_up=8):
    """Auto-detect the repository root; never depends on the CWD being the notebook dir."""
    cur = os.path.abspath(start or os.getcwd())
    tried = []
    for _ in range(max_up):
        tried.append(cur)
        if os.path.isdir(os.path.join(cur, "ml", "fer", "notebooks")):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise FileNotFoundError(
        "Could not locate the repository root (a directory containing ml/fer/notebooks). "
        f"Directories tried, walking up from the CWD: {tried}"
    )

REPO_ROOT = find_repo_root()
FER_ROOT  = os.path.join(REPO_ROOT, "ml", "fer")
OUT_DIR       = os.path.join(FER_ROOT, "outputs")
PLOT_DIR      = os.path.join(FER_ROOT, "plots")
NB08_PLOT_DIR = os.path.join(PLOT_DIR, "nb08")
MODELS_DIR    = os.path.join(FER_ROOT, "models")
for _d in (OUT_DIR, PLOT_DIR, NB08_PLOT_DIR, MODELS_DIR):
    os.makedirs(_d, exist_ok=True)

print("REPO_ROOT:    ", REPO_ROOT)
print("FER_ROOT:     ", FER_ROOT)
print("CWD:          ", os.getcwd())
print("OUT_DIR:      ", OUT_DIR)
print("PLOT_DIR:     ", PLOT_DIR)
print("NB08_PLOT_DIR:", NB08_PLOT_DIR)
print("MODELS_DIR:   ", MODELS_DIR)

# --- byte-budget tracker (same discipline as notebooks 02-07) ----------------
_WRITTEN_FILES = []

def track_write(path, bucket="artifact"):
    """Record a file this notebook wrote, for the end-of-notebook byte report."""
    size = os.path.getsize(path)
    _WRITTEN_FILES.append((path, size))
    return size

def bucket_bytes(files):
    return int(sum(sz for _, sz in files))


REPO_ROOT:     /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168
FER_ROOT:      /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer
CWD:           /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/notebooks
OUT_DIR:       /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs
PLOT_DIR:      /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots
NB08_PLOT_DIR: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb08
MODELS_DIR:    /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models


In [3]:
"""Kernel / version guard.

A prior project notebook broke because a Windows Py3.13 / Keras 3.13 kernel was selected
instead of the WSL GPU kernel. Fail loudly and early if the wrong interpreter is active.
"""
import platform
import tensorflow as tf
keras = tf.keras

print("sys.executable   :", sys.executable)
print("sys.version      :", sys.version)
print("platform.platform:", platform.platform())
print("tf.__version__   :", tf.__version__)
print("keras.__version__:", keras.__version__)

if not keras.__version__.startswith("3.15"):
    raise RuntimeError(
        f"Wrong kernel: keras.__version__ == {keras.__version__!r}, expected a 3.15.x build. "
        "Select the 'WSL GPU (MaternaLink FER)' kernel (name: maternalink-fer-gpu)."
    )
print("\nkernel guard passed.")


I0000 00:00:1787921284.603281    1184 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787921285.374653    1184 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787921288.124436    1184 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


sys.executable   : /home/yasinduslpredetor/miniconda3/envs/maternalink-fer-gpu/bin/python
sys.version      : 3.11.15 (main, Jun 11 2026, 15:20:16) [GCC 14.3.0]
platform.platform: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.43
tf.__version__   : 2.21.0
keras.__version__: 3.15.1

kernel guard passed.


In [4]:
"""Imports and external-tool discovery.

Every binary is verified before use. No seaborn (matplotlib only). The `run()` helper
wraps subprocess so a missing/hanging tool can never crash the notebook.
"""
import json
import shutil
import subprocess
import platform
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("numpy     :", np.__version__)
print("pandas    :", pd.__version__)
print("matplotlib:", matplotlib.__version__)

# --- adb discovery ----------------------------------------------------------
ADB = shutil.which("adb")
print("\nadb on PATH:", ADB if ADB else "NOT FOUND")
ANDROID_SERIAL = os.environ.get("ANDROID_SERIAL")
print("ANDROID_SERIAL env:", ANDROID_SERIAL if ANDROID_SERIAL else "(unset)")
print("TFLITE_BENCHMARK_MODEL env:", os.environ.get("TFLITE_BENCHMARK_MODEL", "(unset)"))


@dataclass
class ProcResult:
    cmd: str
    returncode: int
    stdout: str
    stderr: str
    timed_out: bool
    error: str = ""


def run(cmd, timeout=120):
    """Run a command, capture everything, never raise. `cmd` is a list."""
    printable = " ".join(cmd)
    print("  $", printable)
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return ProcResult(printable, p.returncode, p.stdout, p.stderr, False)
    except subprocess.TimeoutExpired as e:
        print(f"    [timeout after {timeout}s]")
        return ProcResult(printable, -1, e.stdout or "", e.stderr or "", True, "timeout")
    except FileNotFoundError as e:
        print(f"    [binary not found: {e}]")
        return ProcResult(printable, 127, "", "", False, str(e))
    except OSError as e:
        print(f"    [OSError: {e}]")
        return ProcResult(printable, -1, "", "", False, str(e))


VARIANTS = ["float32", "float16", "dynint8", "fullint8"]
REJECTED_VARIANTS = {"fullint8"}
MODEL_PATHS = {v: os.path.join(MODELS_DIR, f"fer_mobilenetv2_96_{v}.tflite") for v in VARIANTS}
CALIBRATED_KERAS = os.path.join(MODELS_DIR, "fer_mobilenetv2_finetuned_96_calibrated.keras")
NOISE_FLOOR = 0.0065           # project macro-F1 noise floor (nb03/nb04)
TEST_MACRO_F1 = 0.6036         # nb05 frozen
TEST_ACCURACY = 0.6289         # nb05 frozen
CALIBRATION_T = 5.727          # nb07
print("\nmodel paths:")
for v, p in MODEL_PATHS.items():
    print(f"  {v:<9} exists={os.path.isfile(p)}  {p}")


numpy     : 2.4.6
pandas    : 3.0.5
matplotlib: 3.11.1

adb on PATH: NOT FOUND
ANDROID_SERIAL env: (unset)
TFLITE_BENCHMARK_MODEL env: (unset)

model paths:
  float32   exists=True  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models/fer_mobilenetv2_96_float32.tflite
  float16   exists=True  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models/fer_mobilenetv2_96_float16.tflite
  dynint8   exists=True  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models/fer_mobilenetv2_96_dynint8.tflite
  fullint8  exists=True  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models/fer_mobilenetv2_96_fullint8.tflite


## Part 1 — P-12: model size on disk (both modes)

**P-12** (PERFORMANCE_BENCHMARK_PLAN.md §4.3) is the size of the exported TFLite file on
disk, per variant. It is a static file property — fully measurable here with no device.
Values are cross-checked against `nb07_tflite_comparison.csv` and must match exactly.


In [5]:
nb07_cmp = pd.read_csv(os.path.join(OUT_DIR, "nb07_tflite_comparison.csv"))
nb07_cmp_by_variant = nb07_cmp.set_index("variant")
print("nb07_tflite_comparison.csv variants:", list(nb07_cmp["variant"]))

float32_bytes = None
size_rows = []
for v in VARIANTS:
    p = MODEL_PATHS[v]
    assert os.path.isfile(p), f"MISSING model file for {v}: {p}"
    b = os.path.getsize(p)
    if v == "float32":
        float32_bytes = b

for v in VARIANTS:
    p = MODEL_PATHS[v]
    b = os.path.getsize(p)
    nb07_b = int(nb07_cmp_by_variant.loc[v, "file_size_bytes"])
    if b != nb07_b:
        print(f"  !!! SIZE MISMATCH for {v}: on-disk {b:,} vs nb07 {nb07_b:,} — INVESTIGATE")
    assert b == nb07_b, f"P-12 cross-check FAILED for {v}: {b} != {nb07_b}"
    size_rows.append({
        "variant": v,
        "file_size_bytes": b,
        "size_kb": round(b / 1024, 2),
        "size_mb": round(b / (1024 * 1024), 4),
        "compression_ratio_vs_float32": round(float32_bytes / b, 4),
        "rejected": v in REJECTED_VARIANTS,
    })

size_df = pd.DataFrame(size_rows)
print()
print(size_df.to_string(index=False))
print("\nP-12 cross-check against nb07_tflite_comparison.csv: PASSED for all 4 variants.")


nb07_tflite_comparison.csv variants: ['keras_calibrated', 'float32', 'float16', 'dynint8', 'fullint8']

 variant  file_size_bytes  size_kb  size_mb  compression_ratio_vs_float32  rejected
 float32          8956864  8746.94   8.5419                        1.0000     False
 float16          4578244  4470.94   4.3662                        1.9564     False
 dynint8          2572184  2511.90   2.4530                        3.4822     False
fullint8          2776016  2710.95   2.6474                        3.2265      True

P-12 cross-check against nb07_tflite_comparison.csv: PASSED for all 4 variants.


## Part 2 — device-independent model memory & parameters (both modes)

The number below is a **model-level** figure: the sum of the TFLite interpreter's tensor
buffers after `allocate_tensors()`, an approximation of the tensor-arena footprint. It is
**NOT P-14** (peak application memory), which can only be measured with the running
Android app on the device. No app → no P-14.


In [6]:
tensor_spec = json.load(open(os.path.join(OUT_DIR, "nb07_tensor_spec.json")))
EXPECTED_IN_SHAPE  = tensor_spec["input_tensor"]["shape"]    # [1, 96, 96, 3]
EXPECTED_OUT_SHAPE = tensor_spec["output_tensor"]["shape"]   # [1, 7]
CLASS_ORDER = tensor_spec["output_tensor"]["class_order"]
print("nb07 tensor spec: input", EXPECTED_IN_SHAPE, "output", EXPECTED_OUT_SHAPE)
print("class order:", CLASS_ORDER)

mem_rows = []
_failures = []
for v in VARIANTS:
    p = MODEL_PATHS[v]
    try:
        interp = tf.lite.Interpreter(model_path=p)
    except Exception as e:  # noqa: BLE001 - printed then handled below
        print(f"  [{v}] Interpreter construction FAILED: {e}")
        _failures.append(v)
        mem_rows.append({"variant": v, "interpreter_tensor_buffer_bytes": None,
                         "n_tensors": None, "input_shape": None, "output_shape": None,
                         "input_dtype": None, "output_dtype": None})
        continue
    try:
        interp.allocate_tensors()
        details = interp.get_tensor_details()
        # tf.lite.Interpreter tensor-detail dicts have no 'bytes' key in TF 2.21;
        # compute a nominal per-tensor size from shape x dtype itemsize (an
        # over-estimate of the real arena - no buffer sharing - which is why this is
        # labelled 'approx').
        def _tensor_nbytes(d):
            n = 1
            for s in d["shape"]:
                n *= max(int(s), 1)
            return n * np.dtype(d["dtype"]).itemsize
        buf_bytes = int(sum(_tensor_nbytes(d) for d in details))
        inp = interp.get_input_details()[0]
        out = interp.get_output_details()[0]
        in_shape = [int(x) for x in inp["shape"]]
        out_shape = [int(x) for x in out["shape"]]
        assert in_shape == EXPECTED_IN_SHAPE, f"{v}: input shape {in_shape} != {EXPECTED_IN_SHAPE}"
        assert out_shape == EXPECTED_OUT_SHAPE, f"{v}: output shape {out_shape} != {EXPECTED_OUT_SHAPE}"
        in_dtype = np.dtype(inp["dtype"]).name
        out_dtype = np.dtype(out["dtype"]).name
        mem_rows.append({
            "variant": v,
            "interpreter_tensor_buffer_bytes": buf_bytes,
            "n_tensors": len(details),
            "input_shape": in_shape,
            "output_shape": out_shape,
            "input_dtype": in_dtype,
            "output_dtype": out_dtype,
        })
        print(f"  [{v}] tensors={len(details):>4}  tensor-buffer bytes (approx arena / "
              f"tensor buffer footprint) = {buf_bytes:>12,}  in={in_dtype} out={out_dtype}")
    except Exception as e:  # noqa: BLE001
        print(f"  [{v}] allocate/inspect FAILED: {e}")
        _failures.append(v)
        mem_rows.append({"variant": v, "interpreter_tensor_buffer_bytes": None,
                         "n_tensors": None, "input_shape": None, "output_shape": None,
                         "input_dtype": None, "output_dtype": None})

if _failures and len(_failures) == len(VARIANTS):
    raise RuntimeError(f"ALL variants failed interpreter inspection: {_failures}")
if _failures:
    print(f"\nWARNING: partial failure — recorded null for: {_failures}")

mem_df = pd.DataFrame(mem_rows)
print()
print(mem_df.to_string(index=False))
print("\nNote: per nb07 tensor spec int8_variant_note, dynint8/fullint8 keep float32 I/O "
      "in this project's variants — actual dtypes recorded above.")


nb07 tensor spec: input [1, 96, 96, 3] output [1, 7]
class order: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
  [float32] tensors= 181  tensor-buffer bytes (approx arena / tensor buffer footprint) =   14,044,404  in=float32 out=float32
  [float16] tensors= 290  tensor-buffer bytes (approx arena / tensor buffer footprint) =   18,475,976  in=float32 out=float32
  [dynint8] tensors= 226  tensor-buffer bytes (approx arena / tensor buffer footprint) =    7,848,204  in=float32 out=float32
  [fullint8] tensors= 184  tensor-buffer bytes (approx arena / tensor buffer footprint) =    3,672,923  in=float32 out=float32

 variant  interpreter_tensor_buffer_bytes  n_tensors    input_shape output_shape input_dtype output_dtype
 float32                         14044404        181 [1, 96, 96, 3]       [1, 7]     float32      float32
 float16                         18475976        290 [1, 96, 96, 3]       [1, 7]     float32      float32
 dynint8                          7848204 

/home/yasinduslpredetor/miniconda3/envs/maternalink-fer-gpu/lib/python3.11/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [7]:
"""Parameter count.

The temperature-scaling wrapper `fer_mobilenetv2_finetuned_96_calibrated.keras` does NOT
round-trip through Keras deserialization even under the Keras version that wrote it (its
saved config is empty) — see the FINDING cell below. Temperature scaling divides logits by
a scalar and adds ZERO parameters, so the parameter count is exactly that of the base
fine-tuned model. Load the base model instead (notebook 07 loaded it successfully) and
cross-check against the counts already frozen in nb05_test_metrics.json.
"""
BASE_KERAS = os.path.join(MODELS_DIR, "fer_mobilenetv2_finetuned_96.keras")

# ground truth: notebook 05 froze these (models.finetuned) — authoritative
_nb05 = json.load(open(os.path.join(OUT_DIR, "nb05_test_metrics.json")))
_NB05_TOTAL = int(_nb05["models"]["finetuned"]["total_params"])          # 2,266,951
_NB05_TRAINABLE = int(_nb05["models"]["finetuned"]["trainable_params"])  # 1,848,583

PARAM_COUNT = None
PARAM_COUNT_TRAINABLE = None
PARAM_COUNT_SOURCE = None
try:
    if not os.path.isfile(BASE_KERAS):
        raise FileNotFoundError(f"missing: {BASE_KERAS}")
    _kmodel = keras.models.load_model(BASE_KERAS, compile=False)
    PARAM_COUNT = int(_kmodel.count_params())
    PARAM_COUNT_TRAINABLE = int(sum(int(np.prod(w.shape)) for w in _kmodel.trainable_weights))
    PARAM_COUNT_SOURCE = "fer_mobilenetv2_finetuned_96.keras (loaded)"
    del _kmodel
    if PARAM_COUNT != _NB05_TOTAL:
        raise AssertionError(
            f"param-count inconsistency: the loaded base model reports {PARAM_COUNT:,} but "
            f"nb05_test_metrics.json froze {_NB05_TOTAL:,}. This is a real discrepancy — "
            "stop and understand it, do not paper over it."
        )
    print(f"[PASS] base model loaded; count_params()={PARAM_COUNT:,} == nb05 frozen count.")
except AssertionError:
    raise
except Exception as e:  # base .keras also unloadable -> fall back to the frozen figures
    PARAM_COUNT = _NB05_TOTAL
    PARAM_COUNT_TRAINABLE = _NB05_TRAINABLE
    PARAM_COUNT_SOURCE = "nb05_test_metrics.json (base .keras also failed to load)"
    print(f"NOTICE: base model load FAILED ({type(e).__name__}: {e}).")
    print("        Falling back to the parameter counts frozen in nb05_test_metrics.json.")

print()
print(f"total parameters      : {PARAM_COUNT:,}")
print(f"  trainable           : {PARAM_COUNT_TRAINABLE:,}")
print(f"  non-trainable       : {PARAM_COUNT - PARAM_COUNT_TRAINABLE:,}")
print(f"parameter-count source: {PARAM_COUNT_SOURCE}")
print("shared by all 4 TFLite variants (float32/float16/dynint8/fullint8); temperature "
      "scaling adds no parameters.")

# --- MAC / FLOP count -----------------------------------------------------
# Not computed: there is no reliable in-environment method here (keras 3 removed the
# graph-level profilers we would trust, and Layer.output_shape is gone). A hand traced
# conv-arithmetic pass over a full MobileNetV2 + custom head is error-prone and any number
# would be a guess. FLOPs are therefore intentionally left unreported rather than
# fabricated. If needed, compute them with the official TF profiler on the frozen graph
# in a dedicated environment.
FLOPS_ESTIMATE = None
print("\nMAC/FLOP count: NOT COMPUTED (no reliable in-environment method — not guessed).")


I0000 00:00:1787921292.692416    1184 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3617 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


[PASS] base model loaded; count_params()=2,266,951 == nb05 frozen count.

total parameters      : 2,266,951
  trainable           : 1,848,583
  non-trainable       : 418,368
parameter-count source: fer_mobilenetv2_finetuned_96.keras (loaded)
shared by all 4 TFLite variants (float32/float16/dynint8/fullint8); temperature scaling adds no parameters.

MAC/FLOP count: NOT COMPUTED (no reliable in-environment method — not guessed).


### FINDING — the calibrated `.keras` wrapper is not reloadable

`ml/fer/models/fer_mobilenetv2_finetuned_96_calibrated.keras` (the temperature-scaling
wrapper notebook 07 built) fails to load with:

> `TypeError: <class 'keras.src.models.functional.Functional'> could not be deserialized
> properly. config={'class_name': 'Functional', 'config': {}, 'build_config':
> {'input_shape': None}}`

The saved config is empty, so the graph cannot be reconstructed. This is **not** the
Keras-version mismatch recorded in `docs/ml/07_calibration_tflite_findings.md` §8 — the
kernel here is Keras 3.15.1, the exact version that wrote the file. It is a distinct
failure of that specific wrapper model.

**Consequence:** the calibrated model exists in practice **only** as the exported
`.tflite` files (`fer_mobilenetv2_96_{float32,float16,dynint8,fullint8}.tflite`). There is
no reloadable Keras artifact for the calibrated model. This strengthens notebook 07’s
argument for TFLite as the deployment format, and is a limitation to record: any future
work needing the calibrated model in Keras form would have to re-fit temperature scaling
on the base model (`fer_mobilenetv2_finetuned_96.keras`, which loads fine) rather than
reload the wrapper.

Parameter counting above is unaffected — temperature scaling adds no parameters, and the
count is taken from the base model and cross-checked against `nb05_test_metrics.json`.


## Part 3 — mode detection

The branch below decides **Mode A** (device present, benchmark runs) vs **Mode B** (no
device, protocol emitted). Mode A requires ALL of: `adb` on PATH; exactly one device in
state `device`; and a usable `arm64-v8a` `benchmark_model` binary. Anything else → Mode B
with a printed reason. Mode B is the expected result in this environment.


In [8]:
BENCHMARK_URL = (
    "https://storage.googleapis.com/tensorflow-nightly-public/prod/tensorflow/release/"
    "lite/tools/nightly/latest/android_aarch64_benchmark_model"
)
NB08_TOOLS_DIR = os.path.join(OUT_DIR, "nb08_tools")
os.makedirs(NB08_TOOLS_DIR, exist_ok=True)

MODE = "B"
MODE_REASON = ""
SELECTED_SERIAL = None
BENCHMARK_BIN = None


def _adb_devices():
    r = run([ADB, "devices"], timeout=30)
    devs = []
    for line in r.stdout.splitlines():
        line = line.strip()
        if not line or line.startswith("List of devices"):
            continue
        parts = line.split()
        if len(parts) >= 2:
            devs.append((parts[0], parts[1]))
    return devs, r


def _resolve_device():
    global SELECTED_SERIAL
    if ADB is None:
        return False, "adb not on PATH"
    devs, r = _adb_devices()
    if r.error == "timeout":
        return False, "adb devices timed out"
    ready = [s for s, st in devs if st == "device"]
    bad = [(s, st) for s, st in devs if st != "device"]
    if bad:
        print("  non-ready devices:", bad)
    if not ready:
        return False, f"no device in state 'device' (saw: {devs or 'none'})"
    if len(ready) > 1:
        if ANDROID_SERIAL and ANDROID_SERIAL in ready:
            SELECTED_SERIAL = ANDROID_SERIAL
            return True, f"multiple devices; ANDROID_SERIAL={ANDROID_SERIAL} selected"
        return False, (f"multiple ready devices {ready} and ANDROID_SERIAL not set/valid — "
                       "set ANDROID_SERIAL to disambiguate")
    SELECTED_SERIAL = ready[0]
    return True, f"single ready device {ready[0]}"


def _resolve_benchmark_binary():
    env_bin = os.environ.get("TFLITE_BENCHMARK_MODEL")
    if env_bin:
        if os.path.isfile(env_bin) and os.access(env_bin, os.X_OK):
            return env_bin, f"TFLITE_BENCHMARK_MODEL={env_bin}"
        print(f"  TFLITE_BENCHMARK_MODEL set but not a usable executable: {env_bin}")
    local = os.path.join(NB08_TOOLS_DIR, "android_aarch64_benchmark_model")
    if os.path.isfile(local):
        return local, f"previously downloaded binary at {local}"
    # attempt download
    try:
        import urllib.request
        print(f"  downloading benchmark_model:\n    {BENCHMARK_URL}")
        urllib.request.urlretrieve(BENCHMARK_URL, local)
        os.chmod(local, 0o755)
        if os.path.getsize(local) < 100_000:
            return None, f"downloaded file implausibly small ({os.path.getsize(local)} bytes)"
        return local, f"downloaded to {local}"
    except Exception as e:  # noqa: BLE001
        return None, f"download failed: {e}"


print("Resolving device...")
dev_ok, dev_msg = _resolve_device()
print("  ->", dev_msg)

if dev_ok:
    print("Resolving benchmark_model binary...")
    BENCHMARK_BIN, bin_msg = _resolve_benchmark_binary()
    print("  ->", bin_msg)
    if BENCHMARK_BIN is not None:
        MODE = "A"
        MODE_REASON = f"device: {dev_msg}; binary: {bin_msg}"
    else:
        MODE = "B"
        MODE_REASON = f"device present ({dev_msg}) but benchmark binary unavailable: {bin_msg}"
else:
    MODE = "B"
    MODE_REASON = dev_msg

RUN["metrics"]["mode"] = MODE
print(f"\n>>> MODE = {MODE}")
print(f">>> MODE_REASON = {MODE_REASON}")
if MODE == "B":
    print(">>> Mode B is the EXPECTED outcome in this environment and is a valid result.")


Resolving device...
  -> adb not on PATH

>>> MODE = B
>>> MODE_REASON = adb not on PATH
>>> Mode B is the EXPECTED outcome in this environment and is a valid result.


## Part 4A — Mode A path (device present)

Every cell in this section no-ops cleanly in Mode B. Runs only when `MODE == "A"`.


In [9]:
DEVICE_INFO = None
if MODE != "A":
    print("skipped — Mode B")
else:
    def _sh(args, timeout=60):
        base = [ADB]
        if SELECTED_SERIAL:
            base += ["-s", SELECTED_SERIAL]
        return run(base + ["shell"] + args, timeout=timeout)

    _props = {}
    r = _sh(["getprop"])
    # parse `[key]: [value]` lines
    for line in r.stdout.splitlines():
        line = line.strip()
        if line.startswith("[") and "]: [" in line:
            k, _, v = line.partition("]: [")
            _props[k[1:]] = v.rstrip("]")

    want = ["ro.product.manufacturer", "ro.product.model", "ro.product.board",
            "ro.board.platform", "ro.hardware", "ro.build.version.release",
            "ro.build.version.sdk", "ro.product.cpu.abi", "ro.build.type",
            "ro.product.device", "ro.soc.manufacturer", "ro.soc.model"]

    cpuinfo = _sh(["cat", "/proc/cpuinfo"]).stdout
    meminfo = _sh(["cat", "/proc/meminfo"]).stdout
    core_count = sum(1 for ln in cpuinfo.splitlines() if ln.lower().startswith("processor"))
    mem_total_kb = None
    for ln in meminfo.splitlines():
        if ln.startswith("MemTotal:"):
            mem_total_kb = int(ln.split()[1])
            break

    DEVICE_INFO = {k: _props.get(k) for k in want}
    DEVICE_INFO.update({
        "adb_serial": SELECTED_SERIAL,
        "cpu_core_count": core_count,
        "mem_total_kb": mem_total_kb,
        "mem_total_mb": round(mem_total_kb / 1024, 1) if mem_total_kb else None,
        "soc_best_guess": _props.get("ro.soc.model") or _props.get("ro.board.platform")
                          or _props.get("ro.hardware"),
        "all_getprop_captured_count": len(_props),
    })
    print(json.dumps(DEVICE_INFO, indent=2))

    build_type = DEVICE_INFO.get("ro.build.type")
    if build_type not in ("user", None):
        print(f"\nWARNING: ro.build.type == {build_type!r} — not a release/'user' build. "
              "Benchmark numbers from a userdebug/eng build are not representative.")
    if not DEVICE_INFO.get("ro.product.model"):
        print("WARNING: device model not reported — PROJECT_CONTROL 'Physical benchmark "
              "device named' cannot be closed without it.")


skipped — Mode B


In [10]:
device_bench_rows = []
if MODE != "A":
    print("skipped — Mode B")
else:
    def _adb(args, timeout=120):
        base = [ADB]
        if SELECTED_SERIAL:
            base += ["-s", SELECTED_SERIAL]
        return run(base + args, timeout=timeout)

    REMOTE_DIR = "/data/local/tmp"
    REMOTE_BIN = f"{REMOTE_DIR}/android_aarch64_benchmark_model"

    _adb(["push", BENCHMARK_BIN, REMOTE_BIN])
    _adb(["shell", "chmod", "755", REMOTE_BIN])
    remote_graphs = {}
    for v in VARIANTS:
        rg = f"{REMOTE_DIR}/fer_mobilenetv2_96_{v}.tflite"
        _adb(["push", MODEL_PATHS[v], rg])
        remote_graphs[v] = rg

    import re

    def _parse_bench(stdout):
        out = {"init_us": None, "inference_avg_us": None, "inference_std_us": None,
               "inference_min_us": None, "inference_max_us": None,
               "peak_mem_mb": None, "overall_mem_mb": None}
        for ln in stdout.splitlines():
            s = ln.strip()
            m = re.search(r"Inference \(avg\):\s*([\d.]+)", s)
            if m:
                out["inference_avg_us"] = float(m.group(1))
            m = re.search(r"Initialization.*?:\s*([\d.]+)", s)
            if m and out["init_us"] is None:
                out["init_us"] = float(m.group(1))
            m = re.search(r"Inference timings in us:.*?Init:\s*([\d.]+).*?"
                          r"First inference:\s*([\d.]+).*?"
                          r"Warmup \(avg\):\s*([\d.]+).*?Inference \(avg\):\s*([\d.]+)", s)
            if m:
                out["init_us"] = float(m.group(1))
                out["inference_avg_us"] = float(m.group(4))
            m = re.search(r"count=(\d+).*?curr=([\d.]+).*?min=([\d.]+).*?max=([\d.]+)"
                          r".*?avg=([\d.e+]+).*?std=(\d+)", s)
            if m:
                out["inference_min_us"] = float(m.group(3))
                out["inference_max_us"] = float(m.group(4))
                out["inference_avg_us"] = float(m.group(5))
                out["inference_std_us"] = float(m.group(6))
            m = re.search(r"Peak memory footprint \(MB\):\s*.*?=\s*([\d.]+)", s)
            if m:
                out["peak_mem_mb"] = float(m.group(1))
            m = re.search(r"Overall memory.*?in MB.*?=\s*([\d.]+)", s)
            if m:
                out["overall_mem_mb"] = float(m.group(1))
        return out

    for v in VARIANTS:
        for nthreads in (1, 2, 4):
            for xnn in (True, False):
                csvfile = f"{REMOTE_DIR}/prof_{v}_{nthreads}_{int(xnn)}.csv"
                r = _adb(["shell", REMOTE_BIN,
                          f"--graph={remote_graphs[v]}",
                          f"--num_threads={nthreads}",
                          f"--use_xnnpack={'true' if xnn else 'false'}",
                          "--warmup_runs=50", "--num_runs=200",
                          "--enable_op_profiling=false",
                          f"--profiling_output_csv_file={csvfile}"], timeout=300)
                parsed = _parse_bench(r.stdout + "\n" + r.stderr)
                # try to pull per-run values for a real p95
                p95_us = None
                pr = _adb(["shell", "cat", csvfile], timeout=60)
                per_run = []
                for ln in pr.stdout.splitlines():
                    for tok in ln.replace(",", " ").split():
                        try:
                            per_run.append(float(tok))
                        except ValueError:
                            pass
                if len(per_run) >= 20:
                    p95_us = float(np.percentile(per_run, 95))
                    median_us = float(np.median(per_run))
                else:
                    median_us = None  # benchmark_model summary mode does not emit median/p95
                row = {
                    "variant": v, "num_threads": nthreads, "use_xnnpack": xnn,
                    "rejected": v in REJECTED_VARIANTS,
                    "mean_latency_ms": (parsed["inference_avg_us"] / 1000.0)
                                        if parsed["inference_avg_us"] else None,
                    "median_latency_ms": (median_us / 1000.0) if median_us else None,
                    "p95_latency_ms": (p95_us / 1000.0) if p95_us else None,
                    "std_latency_ms": (parsed["inference_std_us"] / 1000.0)
                                       if parsed["inference_std_us"] else None,
                    "min_latency_ms": (parsed["inference_min_us"] / 1000.0)
                                       if parsed["inference_min_us"] else None,
                    "max_latency_ms": (parsed["inference_max_us"] / 1000.0)
                                       if parsed["inference_max_us"] else None,
                    "init_ms": (parsed["init_us"] / 1000.0) if parsed["init_us"] else None,
                    "peak_mem_mb": parsed["peak_mem_mb"],
                    "overall_mem_mb": parsed["overall_mem_mb"],
                    "p95_source": "per-run export" if p95_us is not None
                                  else "null — benchmark_model summary mode emits no p95",
                    "returncode": r.returncode,
                }
                device_bench_rows.append(row)
                print(f"  {v:<9} t={nthreads} xnn={int(xnn)}  "
                      f"mean={row['mean_latency_ms']}  p95={row['p95_latency_ms']}")

    device_bench_df = pd.DataFrame(device_bench_rows)
    print()
    print(device_bench_df.to_string(index=False))


skipped — Mode B


In [11]:
thermal_rows = []
if MODE != "A":
    print("skipped — Mode B")
else:
    # winning config: lowest mean latency among NON-rejected variants (median if available)
    cand = [r for r in device_bench_rows
            if not r["rejected"] and r["mean_latency_ms"] is not None]
    if not cand:
        print("no usable latency rows — skipping thermal check")
    else:
        def _key(r):
            return r["median_latency_ms"] if r["median_latency_ms"] is not None else r["mean_latency_ms"]
        win = min(cand, key=_key)
        print(f"winning config: {win['variant']} threads={win['num_threads']} "
              f"xnnpack={win['use_xnnpack']}")
        wv = win["variant"]
        rg = f"/data/local/tmp/fer_mobilenetv2_96_{wv}.tflite"
        for batch in range(1, 6):
            r = run([ADB] + (["-s", SELECTED_SERIAL] if SELECTED_SERIAL else []) +
                    ["shell", "/data/local/tmp/android_aarch64_benchmark_model",
                     f"--graph={rg}", f"--num_threads={win['num_threads']}",
                     f"--use_xnnpack={'true' if win['use_xnnpack'] else 'false'}",
                     "--warmup_runs=50", "--num_runs=200",
                     "--enable_op_profiling=false"], timeout=300)
            import re
            m = re.search(r"Inference \(avg\):\s*([\d.]+)", r.stdout + r.stderr)
            avg_ms = float(m.group(1)) / 1000.0 if m else None
            thermal_rows.append({"batch": batch, "variant": wv, "mean_latency_ms": avg_ms})
            print(f"  batch {batch}: {avg_ms} ms")

        thermal_df = pd.DataFrame(thermal_rows)
        vals = [r["mean_latency_ms"] for r in thermal_rows if r["mean_latency_ms"]]
        if len(vals) >= 2 and vals[0]:
            pct = 100.0 * (vals[-1] - vals[0]) / vals[0]
            print(f"\nbatch1 -> batch5 change: {pct:+.1f}%")
            if pct > 10.0:
                print("#" * 70)
                print("# THERMAL ESCALATION — latency drifted >10% across back-to-back runs.")
                print("# Device is throttling under sustained load. Escalate to ML supervisor.")
                print("#" * 70)
            RUN["metrics"]["thermal_drift_pct_batch1_to_batch5"] = pct

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot([r["batch"] for r in thermal_rows], vals, "o-")
        ax.set_xlabel("consecutive batch (each >=200 timed runs)")
        ax.set_ylabel("mean inference latency (ms)")
        ax.set_title(f"Thermal trend — {wv} (device: sustained back-to-back load)")
        ax.grid(True, alpha=0.3)
        _p = os.path.join(NB08_PLOT_DIR, "thermal_trend.png")
        fig.savefig(_p, dpi=150, bbox_inches="tight")
        plt.close(fig)
        track_write(_p)
        print("saved:", _p)


skipped — Mode B


In [12]:
if MODE != "A":
    print("skipped — Mode B")
else:
    base = [ADB] + (["-s", SELECTED_SERIAL] if SELECTED_SERIAL else [])
    for v in VARIANTS:
        run(base + ["shell", "rm", "-f", f"/data/local/tmp/fer_mobilenetv2_96_{v}.tflite"])
    run(base + ["shell", "rm", "-f", "/data/local/tmp/android_aarch64_benchmark_model"])
    run(base + ["shell", "rm", "-f"] +
        [f"/data/local/tmp/prof_{v}_{t}_{x}.csv"
         for v in VARIANTS for t in (1, 2, 4) for x in (0, 1)])
    leftover = run(base + ["shell", "ls", "/data/local/tmp/"])
    remaining = [ln for ln in leftover.stdout.splitlines()
                 if "fer_mobilenetv2_96_" in ln or "android_aarch64_benchmark_model" in ln
                 or ln.startswith("prof_")]
    if remaining:
        print("WARNING: files still present after cleanup:", remaining)
    else:
        print("cleanup verified — no pushed files remain in /data/local/tmp/")


skipped — Mode B


In [13]:
if MODE != "A":
    print("skipped — Mode B (Mode A artifacts written in the Mode B / common cells below)")
else:
    _p = os.path.join(OUT_DIR, "nb08_device_benchmark.csv")
    pd.DataFrame(device_bench_rows).to_csv(_p, index=False)
    track_write(_p)
    print("saved:", _p)

    dev_record = {
        "mode": "A",
        "reason": MODE_REASON,
        "adb_found": ADB is not None,
        "benchmark_binary": BENCHMARK_BIN,
        "device": DEVICE_INFO,
        "measured_on_device": True,
        "thermal_trend": thermal_rows,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    _p = os.path.join(OUT_DIR, "nb08_device_info.json")
    with open(_p, "w", encoding="utf-8") as f:
        json.dump(dev_record, f, indent=2)
    track_write(_p)
    print("saved:", _p)

    # latency plot
    df = pd.DataFrame(device_bench_rows)
    fig, ax = plt.subplots(figsize=(9, 5))
    for v in VARIANTS:
        sub = df[(df["variant"] == v) & (df["use_xnnpack"])]
        if len(sub):
            ax.plot(sub["num_threads"], sub["mean_latency_ms"], "o-",
                    label=f"{v}{' (REJECTED)' if v in REJECTED_VARIANTS else ''}")
    ax.set_xlabel("num_threads")
    ax.set_ylabel("mean inference latency (ms) — XNNPACK on")
    ax.set_title("Device latency by variant and thread count (P-02, on-device)")
    ax.set_xticks([1, 2, 4])
    ax.legend()
    ax.grid(True, alpha=0.3)
    _p = os.path.join(NB08_PLOT_DIR, "latency_by_variant_and_threads.png")
    fig.savefig(_p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    track_write(_p)
    print("saved:", _p)


skipped — Mode B (Mode A artifacts written in the Mode B / common cells below)


## Part 4B — Mode B path (no device)

Runs only when `MODE == "B"`. Emits the LOUD "not measured" banner, writes the explicit
"no device" record, and generates the device-benchmark protocol (`.md` + `.sh`) for
whoever runs the physical device later.


In [14]:
if MODE != "B":
    print("skipped — Mode A")
else:
    bar = "=" * 78
    print(bar)
    print("  ON-DEVICE METRICS NOT MEASURED")
    print(bar)
    print("  No physical Android device / benchmark_model binary available.")
    print(f"  Reason: {MODE_REASON}")
    print()
    print("  P-02 (FER inference latency) remains UNFILLED.")
    print("  P-01 (face detection), P-03 (preprocessing), P-04 (end-to-end camera->mood)")
    print("  and P-14 (peak app memory) require the Android application and are OUT OF")
    print("  SCOPE for this notebook.")
    print()
    print("  No x86 / development-machine numbers are substituted for device latency.")
    print("  The Proposal's 3-second end-to-end figure is a design target, NOT a measured")
    print("  result, and is not reported as one.")
    print()
    print("  Phase-3 exit remains BLOCKED on P-02.")
    print(bar)


  ON-DEVICE METRICS NOT MEASURED
  No physical Android device / benchmark_model binary available.
  Reason: adb not on PATH

  P-02 (FER inference latency) remains UNFILLED.
  P-01 (face detection), P-03 (preprocessing), P-04 (end-to-end camera->mood)
  and P-14 (peak app memory) require the Android application and are OUT OF
  SCOPE for this notebook.

  No x86 / development-machine numbers are substituted for device latency.
  The Proposal's 3-second end-to-end figure is a design target, NOT a measured
  result, and is not reported as one.

  Phase-3 exit remains BLOCKED on P-02.


In [15]:
if MODE != "B":
    print("skipped — Mode A")
else:
    dev_record = {
        "mode": "B",
        "reason": MODE_REASON,
        "adb_found": ADB is not None,
        "benchmark_binary": None,
        "device": None,
        "measured_on_device": False,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    _p = os.path.join(OUT_DIR, "nb08_device_info.json")
    with open(_p, "w", encoding="utf-8") as f:
        json.dump(dev_record, f, indent=2)
    track_write(_p)
    print("saved:", _p)
    print(json.dumps(dev_record, indent=2))


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb08_device_info.json
{
  "mode": "B",
  "reason": "adb not on PATH",
  "adb_found": false,
  "benchmark_binary": null,
  "device": null,
  "measured_on_device": false,
  "timestamp": "2026-08-28 18:18:16"
}


In [16]:
PROTOCOL_MD = r"""# NB08 Device Benchmark Protocol (P-02 on a physical Android device)

This notebook ran in **Mode B** (no device). P-02 (FER inference latency) is still OPEN.
Follow this checklist on the NAMED physical Android device, then paste the numbers back
and re-run notebook 08's decision-table cell (or hand them to the ML supervisor).

Never report emulator or development-machine timings as device results. Never recommend
`fullint8`. Do not extrapolate device latency from x86 numbers.

## 1. adb setup
1. On the device: Settings -> About phone -> tap Build number 7x to unlock Developer
   Options.
2. Settings -> Developer options -> enable **USB debugging**.
3. Connect USB. Run `adb devices`. Accept the RSA authorisation prompt on the device.
4. Confirm the device shows state `device` (not `unauthorized` / `offline`):
   `adb devices`
5. If more than one device/emulator is attached, set `export ANDROID_SERIAL=<serial>`.
6. Confirm it is a release build: `adb shell getprop ro.build.type` must be `user`.

## 2. benchmark_model binary (arm64-v8a)
Canonical source (VERIFY against current TensorFlow docs at run time — this URL is a
nightly "latest" pointer, not a pinned release):

    https://storage.googleapis.com/tensorflow-nightly-public/prod/tensorflow/release/lite/tools/nightly/latest/android_aarch64_benchmark_model

Fallback — build from source:

    bazel build -c opt --config=android_arm64 //tensorflow/lite/tools/benchmark:benchmark_model

Do NOT invent a version-pinned URL you cannot verify.

## 3. push
    adb push fer_mobilenetv2_96_float32.tflite  /data/local/tmp/
    adb push fer_mobilenetv2_96_float16.tflite  /data/local/tmp/
    adb push fer_mobilenetv2_96_dynint8.tflite  /data/local/tmp/
    adb push fer_mobilenetv2_96_fullint8.tflite /data/local/tmp/
    adb push android_aarch64_benchmark_model    /data/local/tmp/
    adb shell chmod 755 /data/local/tmp/android_aarch64_benchmark_model

## 4. exact invocations — full matrix
4 variants x num_threads {1,2,4} x use_xnnpack {0,1}, warmup >=50, timed >=200:

    for V in float32 float16 dynint8 fullint8; do
      for T in 1 2 4; do
        for X in true false; do
          adb shell /data/local/tmp/android_aarch64_benchmark_model \
            --graph=/data/local/tmp/fer_mobilenetv2_96_${V}.tflite \
            --num_threads=${T} --use_xnnpack=${X} \
            --warmup_runs=50 --num_runs=200 \
            --enable_op_profiling=false \
            --profiling_output_csv_file=/data/local/tmp/prof_${V}_${T}_${X}.csv
        done
      done
    done

Then the thermal run of the WINNER (lowest median latency among float32/float16/dynint8 —
never fullint8), 5 back-to-back batches of >=200 runs each:

    for B in 1 2 3 4 5; do
      adb shell /data/local/tmp/android_aarch64_benchmark_model \
        --graph=/data/local/tmp/fer_mobilenetv2_96_<WINNER>.tflite \
        --num_threads=<WINNER_THREADS> --use_xnnpack=<WINNER_XNN> \
        --warmup_runs=50 --num_runs=200 --enable_op_profiling=false
    done

## 5. what to record — for EVERY combination
- inference latency (ms): mean, median, p95 (from the per-run CSV export), min, max
- init time (ms)
- peak memory footprint / arena footprint (MB)
- device identity block:
  ro.product.manufacturer, ro.product.model, ro.product.board, ro.board.platform,
  ro.hardware, ro.soc.manufacturer, ro.soc.model (SoC), ro.build.version.release
  (Android version), ro.build.version.sdk (API level), ro.product.cpu.abi (ABI),
  ro.build.type (MUST be user/release), core count (/proc/cpuinfo),
  MemTotal (/proc/meminfo)
- battery level and thermal state at start and end
- thermal drift: median latency per batch, % change batch1 -> batch5 (escalate if >10%)

This closes PROJECT_CONTROL open item "Physical benchmark device named".

## 6. cleanup
    adb shell rm -f /data/local/tmp/fer_mobilenetv2_96_*.tflite
    adb shell rm -f /data/local/tmp/android_aarch64_benchmark_model
    adb shell rm -f /data/local/tmp/prof_*.csv
    adb shell ls /data/local/tmp/    # verify nothing of ours remains

## 7. hand-off
Paste the resulting numbers into notebook 08's decision-table cell and re-run it, or hand
them to the ML supervisor. P-02 stays OPEN and Phase-3 exit stays blocked until this is
done on a real device.
"""

PROTOCOL_SH = r"""#!/usr/bin/env bash
# NB08 device benchmark — P-02 on a physical Android device (arm64-v8a).
# Mode B fallback script. Run from a machine with adb + the arm64 benchmark_model binary.
# Never report emulator/x86 timings as device results. Never recommend fullint8.
set -euo pipefail

MODELS_DIR="${MODELS_DIR:-$(cd "$(dirname "$0")/../models" && pwd)}"
BENCH_BIN="${TFLITE_BENCHMARK_MODEL:?set TFLITE_BENCHMARK_MODEL to the arm64 benchmark_model binary}"
REMOTE=/data/local/tmp
VARIANTS=(float32 float16 dynint8 fullint8)
OUT="${1:-nb08_device_benchmark_raw.txt}"

# --- 1. device check ---
adb devices
STATE=$(adb get-state 2>/dev/null || true)
if [ "$STATE" != "device" ]; then echo "no authorised device (state=$STATE)"; exit 1; fi
adb shell getprop ro.build.type
adb shell getprop ro.product.model

# --- 2. device identity block ---
{
  echo "=== DEVICE IDENTITY ==="
  for K in ro.product.manufacturer ro.product.model ro.product.board ro.board.platform \
           ro.hardware ro.soc.manufacturer ro.soc.model ro.build.version.release \
           ro.build.version.sdk ro.product.cpu.abi ro.build.type; do
    echo "$K = $(adb shell getprop $K | tr -d '\r')"
  done
  echo "cpu_cores = $(adb shell cat /proc/cpuinfo | grep -c ^processor)"
  adb shell cat /proc/meminfo | grep MemTotal
} | tee "$OUT"

# --- 3. push ---
for V in "${VARIANTS[@]}"; do
  adb push "${MODELS_DIR}/fer_mobilenetv2_96_${V}.tflite" "${REMOTE}/"
done
adb push "$BENCH_BIN" "${REMOTE}/android_aarch64_benchmark_model"
adb shell chmod 755 "${REMOTE}/android_aarch64_benchmark_model"

# --- 4. full matrix: 4 variants x threads {1,2,4} x xnnpack {true,false} ---
for V in "${VARIANTS[@]}"; do
  for T in 1 2 4; do
    for X in true false; do
      echo "=== ${V} threads=${T} xnnpack=${X} ===" | tee -a "$OUT"
      adb shell "${REMOTE}/android_aarch64_benchmark_model" \
        --graph="${REMOTE}/fer_mobilenetv2_96_${V}.tflite" \
        --num_threads="${T}" --use_xnnpack="${X}" \
        --warmup_runs=50 --num_runs=200 \
        --enable_op_profiling=false \
        --profiling_output_csv_file="${REMOTE}/prof_${V}_${T}_${X}.csv" 2>&1 | tee -a "$OUT"
      adb pull "${REMOTE}/prof_${V}_${T}_${X}.csv" "./prof_${V}_${T}_${X}.csv" || true
    done
  done
done

# --- 5. thermal run of the winner (edit WINNER/WT/WX after inspecting the matrix) ---
WINNER="${WINNER:-float16}"; WT="${WT:-4}"; WX="${WX:-true}"
if [ "$WINNER" = "fullint8" ]; then echo "refusing: fullint8 is REJECTED"; exit 1; fi
for B in 1 2 3 4 5; do
  echo "=== THERMAL batch ${B} : ${WINNER} threads=${WT} xnnpack=${WX} ===" | tee -a "$OUT"
  adb shell "${REMOTE}/android_aarch64_benchmark_model" \
    --graph="${REMOTE}/fer_mobilenetv2_96_${WINNER}.tflite" \
    --num_threads="${WT}" --use_xnnpack="${WX}" \
    --warmup_runs=50 --num_runs=200 --enable_op_profiling=false 2>&1 | tee -a "$OUT"
done

# --- 6. cleanup ---
adb shell rm -f ${REMOTE}/fer_mobilenetv2_96_*.tflite
adb shell rm -f ${REMOTE}/android_aarch64_benchmark_model
adb shell rm -f ${REMOTE}/prof_*.csv
adb shell ls ${REMOTE}/

echo "done — raw output in $OUT. Paste numbers back into notebook 08's decision table."
"""

# always safe to write; spec routes it through the Mode B path
_LF = chr(10)
_p = os.path.join(OUT_DIR, "nb08_device_benchmark_protocol.md")
with open(_p, "w", encoding="utf-8", newline=_LF) as f:
    f.write(PROTOCOL_MD)
track_write(_p)
print("saved:", _p)

_p = os.path.join(OUT_DIR, "nb08_device_benchmark_protocol.sh")
with open(_p, "w", encoding="utf-8", newline=_LF) as f:
    f.write(PROTOCOL_SH)
track_write(_p)
print("saved:", _p)


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb08_device_benchmark_protocol.md
saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb08_device_benchmark_protocol.sh


## Part 6 — P-12 formal record + plots (both modes)


In [17]:
mem_by_variant = {r["variant"]: r for r in mem_rows}
param_by_variant = {v: PARAM_COUNT for v in VARIANTS}

rows = []
for r in size_rows:
    v = r["variant"]
    m = mem_by_variant.get(v, {})
    rows.append({
        "variant": v,
        "file_size_bytes": r["file_size_bytes"],
        "size_mb": r["size_mb"],
        "compression_ratio_vs_float32": r["compression_ratio_vs_float32"],
        "interpreter_tensor_buffer_bytes": m.get("interpreter_tensor_buffer_bytes"),
        "param_count": param_by_variant[v],
        "input_shape": m.get("input_shape"),
        "output_shape": m.get("output_shape"),
        "input_dtype": m.get("input_dtype"),
        "output_dtype": m.get("output_dtype"),
        "rejected": v in REJECTED_VARIANTS,
    })
model_size_mem_df = pd.DataFrame(rows)
_p = os.path.join(OUT_DIR, "nb08_model_size_and_memory.csv")
model_size_mem_df.to_csv(_p, index=False)
track_write(_p)
print("saved:", _p)
print(model_size_mem_df.to_string(index=False))


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb08_model_size_and_memory.csv
 variant  file_size_bytes  size_mb  compression_ratio_vs_float32  interpreter_tensor_buffer_bytes  param_count    input_shape output_shape input_dtype output_dtype  rejected
 float32          8956864   8.5419                        1.0000                         14044404      2266951 [1, 96, 96, 3]       [1, 7]     float32      float32     False
 float16          4578244   4.3662                        1.9564                         18475976      2266951 [1, 96, 96, 3]       [1, 7]     float32      float32     False
 dynint8          2572184   2.4530                        3.4822                          7848204      2266951 [1, 96, 96, 3]       [1, 7]     float32      float32     False
fullint8          2776016   2.6474                        3.2265                          3672923      2266951 [1, 96, 96, 3]       [1, 7]     float32      float32      True


In [18]:
fig, ax = plt.subplots(figsize=(7, 4.5))
colors = ["#c0392b" if v in REJECTED_VARIANTS else "#2c7fb8" for v in size_df["variant"]]
bars = ax.bar(size_df["variant"], size_df["size_mb"], color=colors)
for b, v, mb in zip(bars, size_df["variant"], size_df["size_mb"]):
    ax.text(b.get_x() + b.get_width() / 2, mb + 0.1, f"{mb:.2f} MB",
            ha="center", va="bottom", fontsize=9)
    if v in REJECTED_VARIANTS:
        ax.text(b.get_x() + b.get_width() / 2, mb / 2, "REJECTED",
                ha="center", va="center", rotation=90, color="white", fontweight="bold")
ax.set_ylabel("TFLite file size (MB)")
ax.set_title("P-12 — TFLite model file size on disk, by variant")
ax.grid(True, axis="y", alpha=0.3)
_p = os.path.join(NB08_PLOT_DIR, "size_by_variant.png")
fig.savefig(_p, dpi=150, bbox_inches="tight")
plt.close(fig)
track_write(_p)
print("saved:", _p)


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb08/size_by_variant.png


In [19]:
nb07_parity = pd.read_csv(os.path.join(OUT_DIR, "nb07_parity_check.csv")).set_index("variant")

f32_f1 = float(nb07_cmp_by_variant.loc["float32", "macro_f1"])
fig, ax = plt.subplots(figsize=(8, 5))
ax.axhspan(f32_f1 - NOISE_FLOOR, f32_f1 + NOISE_FLOOR, color="#2ca02c", alpha=0.12,
           label=f"float32 macro-F1 +/- noise floor ({NOISE_FLOOR})")
for v in VARIANTS:
    x = float(nb07_cmp_by_variant.loc[v, "file_size_bytes"]) / (1024 * 1024)
    y = float(nb07_cmp_by_variant.loc[v, "macro_f1"])
    agree = float(nb07_parity.loc[v, "argmax_agreement_rate"]) * 100.0
    rej = v in REJECTED_VARIANTS
    ax.scatter([x], [y], s=140, marker="X" if rej else "o",
               color="#c0392b" if rej else "#2c7fb8", zorder=3,
               edgecolors="black", linewidths=0.5)
    ax.annotate(f"{v}{' (REJECTED)' if rej else ''}\n{agree:.1f}% argmax agree",
                (x, y), textcoords="offset points", xytext=(8, 6), fontsize=8)
ax.set_xlabel("TFLite file size (MB)")
ax.set_ylabel("macro-F1 (VALIDATION, from nb07)")
ax.set_title("Accuracy vs size — accuracy axis is VALIDATION (nb07); latency axis UNMEASURED")
ax.legend(loc="lower right", fontsize=8)
ax.grid(True, alpha=0.3)
_p = os.path.join(NB08_PLOT_DIR, "accuracy_vs_size_tradeoff.png")
fig.savefig(_p, dpi=150, bbox_inches="tight")
plt.close(fig)
track_write(_p)
print("saved:", _p)


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb08/accuracy_vs_size_tradeoff.png


## Part 7 — decision table + recommendation (both modes)

The table combines nb07's accuracy / parity (**VALIDATION**) with the P-12 size measured
here and whatever latency evidence exists (Mode A only). In Mode B the latency column is
`UNMEASURED` and the recommendation rests on **accuracy + size + parity only**.


In [20]:
dev_lat = {}
if MODE == "A":
    _df = pd.DataFrame(device_bench_rows)
    for v in VARIANTS:
        sub = _df[_df["variant"] == v]
        med = sub["median_latency_ms"].dropna()
        p95 = sub["p95_latency_ms"].dropna()
        mean = sub["mean_latency_ms"].dropna()
        dev_lat[v] = {
            "median": float(med.min()) if len(med) else (float(mean.min()) if len(mean) else None),
            "p95": float(p95.min()) if len(p95) else None,
        }

rows = []
for v in VARIANTS:
    delta = float(nb07_cmp_by_variant.loc[v, "macro_f1_delta_vs_keras"])
    agree = float(nb07_parity.loc[v, "argmax_agreement_rate"]) * 100.0
    d = dev_lat.get(v, {})
    rows.append({
        "variant": v,
        "size_mb": next(r["size_mb"] for r in size_rows if r["variant"] == v),
        "macro_f1_val": round(float(nb07_cmp_by_variant.loc[v, "macro_f1"]), 4),
        "macro_f1_delta_vs_keras": round(delta, 4),
        "within_noise_floor": bool(abs(delta) <= NOISE_FLOOR),
        "argmax_agreement_vs_keras_pct": round(agree, 1),
        "ece_val": round(float(nb07_cmp_by_variant.loc[v, "ece"]), 4),
        "device_median_latency_ms": (round(d["median"], 3) if d.get("median") is not None
                                     else "UNMEASURED"),
        "device_p95_latency_ms": (round(d["p95"], 3) if d.get("p95") is not None
                                  else "UNMEASURED"),
        "rejected": v in REJECTED_VARIANTS,
        "recommended": False,
    })

RECOMMENDED = "float16"
for r in rows:
    r["recommended"] = (r["variant"] == RECOMMENDED)

decision_df = pd.DataFrame(rows)
_p = os.path.join(OUT_DIR, "nb08_decision_table.csv")
decision_df.to_csv(_p, index=False)
track_write(_p)
print("saved:", _p)
print(decision_df.to_string(index=False))


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb08_decision_table.csv
 variant  size_mb  macro_f1_val  macro_f1_delta_vs_keras  within_noise_floor  argmax_agreement_vs_keras_pct  ece_val device_median_latency_ms device_p95_latency_ms  rejected  recommended
 float32   8.5419        0.6122                   0.0005                True                           99.5   0.0099               UNMEASURED            UNMEASURED     False        False
 float16   4.3662        0.6090                  -0.0027                True                           99.0   0.0135               UNMEASURED            UNMEASURED     False         True
 dynint8   2.4530        0.6071                  -0.0046                True                           91.5   0.0103               UNMEASURED            UNMEASURED     False        False
fullint8   2.6474        0.4929                  -0.1188               False                           74.5   0.1443               UNMEASURED           

### Recommendation

**`fullint8` is REJECTED and must never be recommended.** nb07: macro-F1 delta vs Keras
is **−0.1188** (≈18× the 0.0065 noise floor), ECE **0.1443**, and only **74.5%** per-sample
argmax agreement with the validated model — and at 2.78 MB it is *larger* than dynint8
(2.57 MB). It is strictly dominated.

Among **float32 / float16 / dynint8** (all within the macro-F1 noise floor):

- **Mode B (this run): recommend `float16`.** The latency axis is **UNMEASURED**, so the
  recommendation rests on accuracy + size + parity only:
  - 4.58 MB — trivially small on any modern Android device;
  - macro-F1 within the noise floor of the validated model;
  - **99.0%** per-sample argmax agreement with the validated model, vs dynint8's **91.5%**
    (dynint8 flips ~1 prediction in 12 — meaningful for per-frame FER with temporal
    smoothing);
  - fp16 is the variant most likely to be fastest on ARM (native hardware fp16).
  - The ~2 MB over dynint8 is negligible. This **mirrors nb07's recommendation.**
- **Mode A:** re-weigh with the measured latency. If `float16` median is within ~1.3× of
  the fastest non-rejected variant, keep `float16`; otherwise present the tradeoff
  explicitly and still do not freeze.

**This is a recommendation, not a freeze — the variant freeze is a Tech Lead decision
after this notebook.**


In [21]:
best_non_rej = None
if MODE == "A" and dev_lat:
    cand = {v: dev_lat[v]["median"] for v in dev_lat
            if v not in REJECTED_VARIANTS and dev_lat[v].get("median") is not None}
    if cand:
        best_non_rej = min(cand, key=cand.get)

print("RECOMMENDED VARIANT:", RECOMMENDED, "(NOT a freeze — Tech Lead decides)")
print()
print("  fullint8 : REJECTED — macro-F1 -0.1188 (~18x noise floor), ECE 0.1443,")
print("             74.5% argmax agreement, larger than dynint8. Never recommended.")
print("  float16  : within noise floor; 99.0% argmax agreement vs validated model;")
print("             4.58 MB; fastest-likely on ARM (hw fp16). Mirrors nb07.")
print("  dynint8  : within noise floor but only 91.5% argmax agreement (~1 in 12 flips).")
print("  float32  : safest accuracy/parity but 2x the size of float16, no accuracy gain.")
if MODE == "A":
    print()
    if best_non_rej:
        fastest = min(dev_lat[v]["median"] for v in dev_lat
                      if v not in REJECTED_VARIANTS and dev_lat[v].get("median"))
        f16 = dev_lat.get("float16", {}).get("median")
        print(f"  Mode A: fastest non-rejected = {best_non_rej} @ {fastest:.3f} ms median;")
        if f16:
            ratio = f16 / fastest
            print(f"          float16 @ {f16:.3f} ms  ->  {ratio:.2f}x fastest")
            if ratio <= 1.3:
                print("          within 1.3x -> keep float16.")
            else:
                print("          >1.3x -> present tradeoff explicitly; do NOT freeze.")
else:
    print()
    print("  Mode B: latency UNMEASURED. Recommendation rests on accuracy + size + parity.")


RECOMMENDED VARIANT: float16 (NOT a freeze — Tech Lead decides)

  fullint8 : REJECTED — macro-F1 -0.1188 (~18x noise floor), ECE 0.1443,
             74.5% argmax agreement, larger than dynint8. Never recommended.
  float16  : within noise floor; 99.0% argmax agreement vs validated model;
             4.58 MB; fastest-likely on ARM (hw fp16). Mirrors nb07.
  dynint8  : within noise floor but only 91.5% argmax agreement (~1 in 12 flips).
  float32  : safest accuracy/parity but 2x the size of float16, no accuracy gain.

  Mode B: latency UNMEASURED. Recommendation rests on accuracy + size + parity.


## Part 8 — Phase-3 metric status (both modes)

| Metric | Status | Value | What is required to close it |
|---|---|---|---|
| **P-01** face detection | OPEN | — | The Android application (does not exist yet) |
| **P-02** FER inference latency | OPEN unless Mode A ran | — (Mode B) | Physical Android device + `benchmark_model` (Mode A of this notebook) |
| **P-03** preprocessing latency | OPEN | — | The Android application (does not exist yet) |
| **P-04** end-to-end camera→mood | OPEN | — | The Android application (does not exist yet) |
| **P-12** TFLite model size on disk | **CLOSED (this notebook)** | float32 8.96 MB · float16 4.58 MB · dynint8 2.57 MB · fullint8 2.78 MB (REJECTED) | Done |
| **P-14** peak app memory | OPEN | — | The Android application (does not exist yet) |

PROJECT_CONTROL open item **"Physical benchmark device named"** — CLOSED only if this
notebook ran in Mode A and recorded a real device in `nb08_device_info.json`; otherwise
still OPEN. The interpreter tensor-buffer footprint reported in Part 2 is a model-level
approximation and is **not** P-14.

The Proposal's 3-second end-to-end target is a design target and is **not** reported as a
measured value anywhere in this notebook.


## Part 9 — save_run


In [22]:
RUN["model_version"] = "fer_mobilenetv2_96_{float32,float16,dynint8,fullint8}.tflite"

RUN["hyperparameters"] = {
    "variants": VARIANTS,
    "rejected_variants": sorted(REJECTED_VARIANTS),
    "benchmark_matrix": {
        "num_threads": [1, 2, 4],
        "use_xnnpack": [True, False],
        "warmup_runs": 50,
        "num_runs": 200,
        "thermal_batches": 5,
    },
    "benchmark_binary_url": BENCHMARK_URL,
    "noise_floor_macro_f1": NOISE_FLOOR,
    "reads_labelled_images": False,
    "test_set_touch_count": 0,
}

RUN["metrics"]["mode"] = MODE
RUN["metrics"]["mode_reason"] = MODE_REASON
RUN["metrics"]["test_set_touch_count"] = 0
RUN["metrics"]["param_count_all_variants"] = PARAM_COUNT
RUN["metrics"]["param_count_source"] = PARAM_COUNT_SOURCE
RUN["metrics"]["param_count_trainable"] = PARAM_COUNT_TRAINABLE
RUN["metrics"]["flops_estimate"] = FLOPS_ESTIMATE
RUN["metrics"]["p12_file_size_bytes"] = {r["variant"]: r["file_size_bytes"] for r in size_rows}
RUN["metrics"]["p12_size_mb"] = {r["variant"]: r["size_mb"] for r in size_rows}
RUN["metrics"]["interpreter_tensor_buffer_bytes"] = {
    r["variant"]: r["interpreter_tensor_buffer_bytes"] for r in mem_rows
}
RUN["metrics"]["p02_device_latency_measured"] = (MODE == "A")
RUN["metrics"]["recommended_variant"] = RECOMMENDED
RUN["metrics"]["recommended_is_freeze"] = False
RUN["metrics"]["carried_from_nb07"] = {
    "test_macro_f1": TEST_MACRO_F1, "test_accuracy": TEST_ACCURACY,
    "calibration_T": CALIBRATION_T, "noise_floor": NOISE_FLOOR,
}
if MODE == "A":
    RUN["metrics"]["device_benchmark_rows"] = device_bench_rows
    RUN["metrics"]["device_info"] = DEVICE_INFO

RUN["notes"] = (
    f"Notebook 08 is a MODEL-level benchmark. Ran in MODE {MODE} ({MODE_REASON}). "
    "P-12 (TFLite file size on disk) recorded for all 4 variants and cross-checked against "
    "nb07_tflite_comparison.csv. "
    + ("On-device FER inference latency (P-02) measured via benchmark_model across "
       "4 variants x threads{1,2,4} x xnnpack{on,off} plus a 5-batch thermal run; "
       "device recorded in nb08_device_info.json. "
       if MODE == "A" else
       "On-device latency (P-02) NOT measured — no physical device / benchmark binary. "
       "P-02 remains OPEN, Phase-3 exit blocked on it; a device-benchmark protocol "
       "(.md + .sh) was emitted. No x86 numbers substituted. ")
    + "P-01/P-03/P-04/P-14 require the Android application (does not exist yet) and are "
    "out of scope here. Recommended variant: float16 (mirrors nb07) — a RECOMMENDATION, "
    "NOT a freeze; the variant freeze is a Tech Lead decision. The Proposal's 3-second "
    "target is a design target, not reported as measured. TEST split untouched "
    "(test_set_touch_count = 0)."
)

written_total = bucket_bytes(_WRITTEN_FILES)
print(f"Total bytes written by this notebook: {written_total:,}")
for _p, _sz in _WRITTEN_FILES:
    print(f"  {_sz:>12,}  {_p}")

print()
print(json.dumps({k: v for k, v in RUN.items() if k != "metrics"}, indent=2, default=str))
save_run()


Total bytes written by this notebook: 116,704
           185  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb08_device_info.json
         4,261  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb08_device_benchmark_protocol.md
         3,142  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb08_device_benchmark_protocol.sh
           538  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb08_model_size_and_memory.csv
        34,424  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb08/size_by_variant.png
        73,655  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb08/accuracy_vs_size_tradeoff.png
           499  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb08_decision_table.csv

{
  "run_id": "run_20260828_181803",
  "timestamp": "2026-08-28 18:18:03",
  "notebook": "08_mobile_benchmark",
  "dataset_version": null,
  "model_version":

'../outputs/run_20260828_181803.json'